# Bolt raw data

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [25]:
# Andmestiku sisselugemine

df = pd.read_csv("bolt_data.csv")

# Esimesed viis rida
df.head()

,order_id_new,order_try_id_new,calc_created,metered_price,upfront_price,distance,duration,gps_confidence,entered_by,b_state,...,device_token,rider_app_version,order_state,order_try_state,driver_app_version,driver_device_uid_new,device_name,eu_indicator,overpaid_ride_ticket,fraud_score
0,22,22,2020-02-02 3:37:31,4.04,10.0,2839,700,1,client,finished,...,NaN,CI.4.17,finished,finished,DA.4.37,1596,Xiaomi Redmi 6,1,0,-1383.0
1,618,618,2020-02-08 2:26:19,6.09,3.6,5698,493,1,client,finished,...,NaN,CA.5.43,finished,finished,DA.4.39,1578,Samsung SM-G965F,1,0,NaN
2,657,657,2020-02-08 11:50:35,4.32,3.5,4426,695,1,client,finished,...,NaN,CA.5.43,finished,finished,DA.4.37,951,Samsung SM-A530F,1,0,-166.0
3,313,313,2020-02-05 6:34:54,72871.72,NaN,49748,1400,0,client,finished,...,NaN,CA.5.23,finished,finished,DA.4.37,1587,TECNO-Y6,0,1,NaN
4,1176,1176,2020-02-13 17:31:24,20032.50,19500.0,10273,5067,1,client,finished,...,NaN,CA.5.04,finished,finished,DA.4.37,433,Itel W5504,0,0,NaN


In [ ]:
# Leian, kas terve "device_token" veerg on tühi või mitte. On jah. 

df["device_token"].isna().sum()

np.int64(4943)

In [27]:
df["device_token"].isna().all()

np.True_

In [ ]:
# Eemaldan veeru "device_token".

df = df.drop(columns=["device_token"])

In [ ]:
# Kuupäevad on string-vormingus. Teisendan kuupäevaks
df["calc_created"] = pd.to_datetime(
    df["calc_created"], format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)
df["calc_created"].head()

0   2020-02-02 03:37:31
1   2020-02-08 02:26:19
2   2020-02-08 11:50:35
3   2020-02-05 06:34:54
4   2020-02-13 17:31:24
Name: calc_created, dtype: datetime64[us]

In [30]:
# Loon puhastatava andmestiku
df_clean = df.copy()

In [ ]:
# puhastan välja aasta, kuu, nädalapäeva ja tunni. igaks juhuks. 
# pärast saab ajaliselt leida seoseid või nende puudumist

df_clean["date"] = df_clean["calc_created"].dt.date
df_clean["time"] = df_clean["calc_created"].dt.time

df_clean["day_of_week"] = df_clean["calc_created"].dt.day_name()
df_clean["day_of_week_nr"] = df_clean["calc_created"].dt.dayofweek + 1

df_clean["month"] = df_clean["calc_created"].dt.month
df_clean["year"] = df_clean["calc_created"].dt.year

df_clean.head()

,order_id_new,order_try_id_new,calc_created,metered_price,upfront_price,distance,duration,gps_confidence,entered_by,b_state,...,device_name,eu_indicator,overpaid_ride_ticket,fraud_score,date,time,day_of_week,day_of_week_nr,month,year
0,22,22,2020-02-02 03:37:31,4.04,10.0,2839,700,1,client,finished,...,Xiaomi Redmi 6,1,0,-1383.0,2020-02-02,03:37:31,Sunday,7,2,2020
1,618,618,2020-02-08 02:26:19,6.09,3.6,5698,493,1,client,finished,...,Samsung SM-G965F,1,0,NaN,2020-02-08,02:26:19,Saturday,6,2,2020
2,657,657,2020-02-08 11:50:35,4.32,3.5,4426,695,1,client,finished,...,Samsung SM-A530F,1,0,-166.0,2020-02-08,11:50:35,Saturday,6,2,2020
3,313,313,2020-02-05 06:34:54,72871.72,NaN,49748,1400,0,client,finished,...,TECNO-Y6,0,1,NaN,2020-02-05,06:34:54,Wednesday,3,2,2020
4,1176,1176,2020-02-13 17:31:24,20032.50,19500.0,10273,5067,1,client,finished,...,Itel W5504,0,0,NaN,2020-02-13,17:31:24,Thursday,4,2,2020


In [ ]:
# kas on täielikult identseid ridu? Ei ole.

df_clean[df_clean.duplicated(keep=False)]

,order_id_new,order_try_id_new,calc_created,metered_price,upfront_price,distance,duration,gps_confidence,entered_by,b_state,...,device_name,eu_indicator,overpaid_ride_ticket,fraud_score,date,time,day_of_week,day_of_week_nr,month,year


In [ ]:
# Andmetüüpide kontroll

df_clean.dtypes

order_id_new                      int64
order_try_id_new                  int64
calc_created             datetime64[us]
metered_price                   float64
upfront_price                   float64
distance                          int64
duration                          int64
gps_confidence                    int64
entered_by                          str
b_state                             str
dest_change_number                int64
prediction_price_type               str
predicted_distance              float64
predicted_duration              float64
change_reason_pricing               str
ticket_id_new                     int64
rider_app_version                   str
order_state                         str
order_try_state                     str
driver_app_version                  str
driver_device_uid_new             int64
device_name                         str
eu_indicator                      int64
overpaid_ride_ticket              int64
fraud_score                     float64


In [42]:
df_clean["order_id_new"].equals(df_clean["order_try_id_new"])

False

In [43]:
(df_clean["order_id_new"] != df_clean["order_try_id_new"]).sum()

np.int64(58)

##### (sain teada, et order_id_new ja order_try_id_new ei ole identsed, id numbrid erinevad 58 korral)

In [ ]:
# vaatan, mis väärtused on veerus "prediction_price_type" ja kui palju neid on

df_clean["prediction_price_type"].value_counts(
    normalize=True,
    dropna=False
) * 100

prediction_price_type
upfront                        69.431519
prediction                     25.874975
upfront_destination_changed     4.207971
NaN                             0.404613
upfront_waypoint_changed        0.080923
Name: proportion, dtype: float64

#### on 20 sõitu, mille puhul on kõik need kategooriad NaN
prediction_price_type

prediction_price_type

upfront_price            

predicted_distance

predicted_duration

change_reason_pricing


In [47]:
cols = [
    "prediction_price_type",
    "upfront_price",
    "predicted_distance",
    "predicted_duration",
    "change_reason_pricing"
]

# Valin read, kus kõik viis väärtust puuduvad
missing_rides = df_clean[
    df_clean[cols].isna().all(axis=1)
]

# Kuvan
missing_rides

,order_id_new,order_try_id_new,calc_created,metered_price,upfront_price,distance,duration,gps_confidence,entered_by,b_state,...,device_name,eu_indicator,overpaid_ride_ticket,fraud_score,date,time,day_of_week,day_of_week_nr,month,year
64,217,217,2020-02-04 08:07:10,NaN,NaN,6249,2477,1,driver,finished,...,Xiaomi Redmi 8,1,0,NaN,2020-02-04,08:07:10,Tuesday,2,2,2020
393,3066,3066,2020-03-02 17:45:17,NaN,NaN,5483,917,1,driver,finished,...,HUAWEI EML-L29,1,0,NaN,2020-03-02,17:45:17,Monday,1,3,2020
458,3320,3320,2020-03-06 03:33:52,NaN,NaN,3364,323,0,reseller,finished,...,HUAWEI CLT-L29,1,0,NaN,2020-03-06,03:33:52,Friday,5,3,2020
513,1166,1166,2020-02-13 14:55:14,NaN,NaN,21997,1742,1,driver,finished,...,HUAWEI SLA-L22,1,0,-74.0,2020-02-13,14:55:14,Thursday,4,2,2020
779,1759,1759,2020-02-19 00:55:07,NaN,NaN,10309,849,1,reseller,finished,...,"iPhone8,1",1,0,NaN,2020-02-19,00:55:07,Wednesday,3,2,2020
998,861,861,2020-02-10 06:52:29,NaN,NaN,1424,416,1,driver,finished,...,Samsung SM-G973F,1,0,NaN,2020-02-10,06:52:29,Monday,1,2,2020
1206,1349,1349,2020-02-14 23:08:22,NaN,NaN,11206,1268,1,client,finished,...,Samsung SM-T295,1,0,-1270.0,2020-02-14,23:08:22,Friday,5,2,2020
1287,2012,2012,2020-02-21 13:24:40,NaN,NaN,3717,713,1,driver,finished,...,Samsung SM-A530F,1,0,NaN,2020-02-21,13:24:40,Friday,5,2,2020
1340,1112,1112,2020-02-12 23:32:11,NaN,NaN,4114,528,1,driver,finished,...,Xiaomi Mi A1,1,0,NaN,2020-02-12,23:32:11,Wednesday,3,2,2020
1582,460,460,2020-02-06 19:34:43,NaN,NaN,6718,764,1,client,finished,...,"iPhone8,1",1,0,NaN,2020-02-06,19:34:43,Thursday,4,2,2020


In [ ]:
# Salvestan puhastatud andmestiku CSV-failina, et saaks teistes notebookides kasutada.

df_clean.to_csv(
    "bolt_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)